In [8]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [2]:
TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\train"
VALID_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\valid"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\test"

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
FINE_TUNE_AT = 249
LR_HEAD = 1e-3


In [4]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)



In [5]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)


valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 11137 images belonging to 12 classes.
Found 240 images belonging to 12 classes.
Found 600 images belonging to 12 classes.


In [6]:
num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print("Classes:", num_classes)
print(class_names)


Classes: 12
['Black Rust', 'Blast', 'Brown Rust', 'Common Root Rot', 'Fusarium Head Blight', 'Healthy', 'Leaf Blight', 'Mildew', 'Septoria', 'Smut', 'Tan spot', 'Yellow Rust']


In [7]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


base_model.trainable = False  # stage 1 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,589,004 (9.88 MB)

 Trainable params: 331,020 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "MobileNetV2_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = model.fit(
    train_generator,
    epochs=EPOCHS_HEAD,
    validation_data=valid_generator,
    callbacks = callbacks
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
110/349 ━━━━━━━━━━━━━━━━━━━━ 4:52 1s/step - accuracy: 0.3627 - loss: 2.0942

KeyboardInterrupt: 

In [10]:
FINE_TUNE_AT =  

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_generator,
    epochs=EPOCHS_FINE,
    validation_data=valid_generator,
    verbose=1
)

Epoch 1/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 889s 2s/step - accuracy: 0.2237 - loss: 2.2550 - val_accuracy: 0.2167 - val_loss: 2.3657
Epoch 2/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 823s 2s/step - accuracy: 0.3810 - loss: 1.9130 - val_accuracy: 0.2833 - val_loss: 2.2907
Epoch 3/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 736s 2s/step - accuracy: 0.4494 - loss: 1.7224 - val_accuracy: 0.3375 - val_loss: 2.2247
Epoch 4/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 720s 2s/step - accuracy: 0.4926 - loss: 1.5992 - val_accuracy: 0.3542 - val_loss: 2.1486
Epoch 5/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 811s 2s/step - accuracy: 0.5073 - loss: 1.5231 - val_accuracy: 0.3625 - val_loss: 2.1024
Epoch 6/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - accuracy: 0.5322 - loss: 1.4562 - val_accuracy: 0.3875 - val_loss: 2.0134
Epoch 7/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 831s 2s/step - accuracy: 0.5591 - loss: 1.3634 - val_accuracy: 0.4208 - val_loss: 1.9428
Epoch 8/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 837s 2s/step - accuracy: 0.5762 - loss: 1.3221 - val_accu

In [11]:
model.save('InceptionV3_WPD.keras')

In [12]:
import json
with open("training_history_InceptionV3.json", "w", encoding="utf-8") as f:
    json.dump(history_head.history, f, indent=2)



Saved: training_history_InceptionV3.json


In [14]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

19/19 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - accuracy: 0.6241 - loss: 1.3918
Test loss: 1.4289
Test accuracy: 0.6467


In [16]:
import numpy as np

y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

19/19 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step 


In [17]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


                           precision    recall  f1-score   support

          black_rust_test       0.86      0.60      0.71        50
               blast_test       0.78      0.86      0.82        50
          brown_rust_test       0.36      0.38      0.37        50
     common_root_rot_test       0.72      0.72      0.72        50
fusarium_head_blight_test       0.93      0.78      0.85        50
             healthy_test       0.10      0.04      0.06        50
         leaf_blight_test       0.67      0.60      0.63        50
              mildew_test       0.60      0.60      0.60        50
            septoria_test       0.73      0.90      0.80        50
                smut_test       0.87      0.96      0.91        50
            tan_spot_test       0.59      0.34      0.43        50
         yellow_rust_test       0.47      0.98      0.64        50

                 accuracy                           0.65       600
                macro avg       0.64      0.65      0.63    

In [21]:


cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=class_names,      
    columns=class_names     

print(cm_df)


                           black_rust_test  blast_test  brown_rust_test  \
black_rust_test                         30           1               12   
blast_test                               0          43                0   
brown_rust_test                          1           0               19   
common_root_rot_test                     0           3                2   
fusarium_head_blight_test                0           0                2   
healthy_test                             1           2                1   
leaf_blight_test                         1           0                6   
mildew_test                              0           1                7   
septoria_test                            0           0                0   
smut_test                                0           1                0   
tan_spot_test                            2           4                3   
yellow_rust_test                         0           0                1   

                        

In [22]:
cm_df

,black_rust_test,blast_test,brown_rust_test,common_root_rot_test,fusarium_head_blight_test,healthy_test,leaf_blight_test,mildew_test,septoria_test,smut_test,tan_spot_test,yellow_rust_test
black_rust_test,30,1,12,1,0,1,2,0,1,0,1,1
blast_test,0,43,0,1,0,2,1,0,0,2,1,0
brown_rust_test,1,0,19,2,0,3,1,11,6,0,0,7
common_root_rot_test,0,3,2,36,0,4,1,3,1,0,0,0
fusarium_head_blight_test,0,0,2,1,39,1,2,0,0,3,0,2
healthy_test,1,2,1,0,0,2,0,0,0,0,2,42
leaf_blight_test,1,0,6,1,1,2,30,1,1,1,5,1
mildew_test,0,1,7,1,0,2,0,30,4,0,3,2
septoria_test,0,0,0,1,0,1,3,0,45,0,0,0
smut_test,0,1,0,0,0,0,1,0,0,48,0,0
